## Notebook 14 — Fixing Finding 10 + Extending Finding 09 to 6 Types

Two independent parts, run in sequence in 1 session (the model is loaded once):

**Part A (fixing finding 10):** notebook 13 picked questions PER PAIR
(top-disagreement between 2 specific groups) -> a median of only 2-3
groups/question, not enough for a probe. Here questions are picked using
**broad coverage** (answered by many/all cells per type, the same logic as
notebook 07/09), then the **L11 AND L1** vectors are extracted (L1 = the
new candidate from finding 09 for RACE) at the opinion-answer position, for
ALL 6 types.

**Part B (extending finding 09):** the 32-layer sweep + random control (the
exact same engine as notebook 13, already verified to pass 5+3 offline
checks) is extended to the 3 types not yet tested causally:
EDUCATIONxINCOME, RACExPOLPARTY, RACExPOLIDEOLOGY.

Not mandatory work for Paper 1 (a design decision, not a hole), but cheap
(reusing an already-tested engine) so it is done along the way.


## Before running: Kaggle setup

1. **Accelerator**: GPU T4 x2. **Internet: On**. Attach `opinionqa_intersectional.csv`.
2. Session left over from a crash -> RESTART SESSION.
3. **Download when finished** from `/kaggle/working/stage2_probefix_sweep6/`:
   `probe_features_v2.csv`, `probe_features_v2.npz` (Part A),
   `sweep6_rows.csv`, `sweep6_summary_raw.csv` (Part B) -> put them in
   `results/08_probe_causal_dissociation/`.

Estimate: model load ~10 min; Part A (plain extraction, 2 layers, many
questions) ~15-20 min; Part B (sweep of 3 new types, same engine as
notebook 13) ~15-20 min. Total ~45 min - 1 hour.


In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy tqdm

In [ ]:
import os, sys, gc, glob, ast
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, wilcoxon
from tqdm.auto import tqdm

sys.last_traceback = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f"GPU {d}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
        if free / total < 0.9:
            print(f"  WARNING: GPU {d} is not empty -> RESTART SESSION first!")


In [ ]:
MODEL_PATH = "mistralai/Mistral-7B-v0.1"

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError("opinionqa_intersectional.csv not found.")
print("Data:", DATA_PATH)

RANDOM_SEED = 42
ALL_TYPES = ["AGExPOLPARTY", "EDUCATIONxINCOME", "RACExRELIG",
             "RACExPOLPARTY", "RACExPOLIDEOLOGY", "RELIGxPOLPARTY"]
TYPES_ALREADY_SWEPT = ["AGExPOLPARTY", "RELIGxPOLPARTY", "RACExRELIG"]  # notebook 13
TYPES_NEW_SWEEP = [t for t in ALL_TYPES if t not in TYPES_ALREADY_SWEPT]  # Part B
MAX_OPTIONS = 6
STAR_LAYER, STAR_HEAD = 11, 16
L1_LAYER = 1  # new candidate from finding 09 (the RACE causal unit)

# demographic labels per type, used to build the QA questions (same as ATTR_LABELS in notebook 09)
ATTR_LABELS = {
    "RACExRELIG":       ("race", "religion"),
    "RACExPOLPARTY":    ("race", "political party affiliation"),
    "RACExPOLIDEOLOGY": ("race", "political ideology"),
    "RELIGxPOLPARTY":   ("religion", "political party affiliation"),
    "EDUCATIONxINCOME": ("highest level of education", "household income"),
    "AGExPOLPARTY":     ("age group", "political party affiliation"),
}
ATTR_QA = {ty: (f"What is this survey respondent's {l1}?",
               f"What is this survey respondent's {l2}?")
          for ty, (l1, l2) in ATTR_LABELS.items()}

OUT_DIR = "/kaggle/working/stage2_probefix_sweep6"
os.makedirs(OUT_DIR, exist_ok=True)


## Shared data & prompt functions (used by Part A and B)

`build_prompt` is exactly the same as notebook 12/13 (identity = a 1-token
QA answer, then the opinion question, ending with `Answer:`).


In [ ]:
df = pd.read_csv(DATA_PATH)
for c in ["responses", "ordinal", "options"]:
    df[c] = df[c].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["group_key"] = df["attribute"] + " :: " + df["group"]
df["n_opt"] = df["ordinal"].apply(len)

qmeta = {}
real_resp = {}
for r in df.itertuples():
    qmeta[r.qkey] = (r.question, r.options[: r.n_opt], r.ordinal)
    real_resp[(r.group_key, r.qkey)] = np.array(r.responses, dtype=np.float64)

def real_wd(A, B, qk):
    _, _, ordinal = qmeta[qk]
    return wasserstein_distance(ordinal, ordinal,
                                u_weights=real_resp[(A, qk)], v_weights=real_resp[(B, qk)])

DEMO_LETTERS = [chr(65 + i) for i in range(26)]
LETTERS = ["A", "B", "C", "D", "E", "F"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def demo_block(question, opts, value):
    lines = [f"Question: {question}"]
    for i, o in enumerate(opts):
        lines.append(f"{DEMO_LETTERS[i]}. {o}")
    lines.append(f"Answer: {DEMO_LETTERS[opts.index(value)]}")
    return "\n".join(lines)

def build_prompt(ty, v1_opts, v2_opts, gk, qk):
    v1, v2 = gk.split(" :: ", 1)[1].split(" | ", 1)
    q1, q2 = ATTR_QA[ty]
    question, options, _ = qmeta[qk]
    blocks = [demo_block(q1, v1_opts, v1), "", demo_block(q2, v2_opts, v2), "",
              f"Question: {question}"]
    for i, opt in enumerate(options):
        blocks.append(f"{LETTERS[i]}) {opt}")
    blocks.append("Answer:")
    return "\n".join(blocks)

def identity_positions(prompt, n_blocks=2):
    positions = []
    search_from = 0
    for _ in range(n_blocks):
        idx = prompt.find("Answer:", search_from)
        assert idx != -1
        prefix = prompt[: idx + len("Answer:")]
        pos = 1 + len(tokenizer.encode(prefix, add_special_tokens=False))
        positions.append(pos)
        search_from = idx + 1
    return positions

def last_token_abs_pos(prompt):
    """Absolute index of the LAST token (opinion-answer position), with BOS.
    seq_len = 1(BOS) + len(tokens without BOS); last index = seq_len - 1
    = len(tokens without BOS). Consistent with the identity_positions() convention."""
    return len(tokenizer.encode(prompt, add_special_tokens=False))

type_opts = {}
for ty in ALL_TYPES:
    sub = df[(df["attribute"] == ty) & (df["n_opt"] <= MAX_OPTIONS)]
    cells = sorted(sub["group_key"].unique().tolist())
    v1_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[0] for gk in cells})
    v2_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[1] for gk in cells})
    type_opts[ty] = dict(cells=cells, v1_opts=v1_opts, v2_opts=v2_opts,
                         q_per_cell=sub.groupby("group_key")["qkey"].apply(set).to_dict())
    print(f"[{ty}] {len(cells)} cells")


## Part A — Data: BROAD COVERAGE questions (fixing finding 10)

Different from notebook 13: questions are picked by **how many cells
answer them**, not by per-pair disagreement. Take every question with
coverage >= 60% of the cells in that type (at least 10), then **randomly
sample at most 250 questions per type** (`MAX_Q_PER_TYPE`).

Why cap it: without the cap, Part A = 5,855 questions / 131 thousand
forward passes (~4x longer, and the final `np.stack` needs ~8.6 GB RAM ->
prone to OOM right at the last second). With the cap: 1,500 questions /
33 thousand forwards, n per type still 4,000-7,200 rows -- far more than
enough for a probe (the finding 10 that failed only had n=8-13).

Why random rather than top-coverage: the highest-coverage questions pile
up in 2-3 survey waves (EDUCATIONxINCOME: the top-250 are only waves
50/54/92), so the probe only learns a handful of topics. A random sample
from the questions that already pass the threshold hits all 15 waves with
nearly the same coverage.


In [ ]:
MIN_COVERAGE_FRAC = 0.6
MIN_COVERAGE_ABS = 10
MAX_Q_PER_TYPE = 250   # cap: 5,855 questions (all) -> 1,500 questions, ~4x faster.
                       # n per type still 4,000-7,200 rows (plenty for a probe;
                       # the failed finding 10 only had n=8-13). Raise it if the GPU allows.

probeA_plan = {}
rng_q = np.random.default_rng(RANDOM_SEED)
for ty in ALL_TYPES:
    info = type_opts[ty]
    cells = info["cells"]
    n_cell = len(cells)
    min_cov = max(MIN_COVERAGE_ABS, int(np.ceil(MIN_COVERAGE_FRAC * n_cell)))
    min_cov = min(min_cov, n_cell)  # never more than the number of cells

    q_count = {}
    for gk in cells:
        for qk in info["q_per_cell"].get(gk, set()):
            q_count[qk] = q_count.get(qk, 0) + 1
    good_qs = sorted([qk for qk, c in q_count.items() if c >= min_cov])

    # IMPORTANT: the cap is RANDOM, not "take the highest coverage". If sorted
    # by coverage and then taking the top-N, the picked questions pile up in just
    # 2-3 waves (checked: EDUCATIONxINCOME top-250 is only waves 50/54/92, while
    # random-250 hits all 15) -- the probe then looks good but only for a few topics.
    if len(good_qs) > MAX_Q_PER_TYPE:
        idx = rng_q.choice(len(good_qs), size=MAX_Q_PER_TYPE, replace=False)
        picked = sorted(good_qs[i] for i in idx)
    else:
        picked = good_qs

    n_rows = sum(q_count[qk] for qk in picked)
    probeA_plan[ty] = dict(cells=cells, good_qs=picked, min_cov=min_cov,
                           n_good_all=len(good_qs), n_rows_est=n_rows)
    print(f"[{ty}] {n_cell} cells, coverage threshold={min_cov}, "
          f"questions passing={len(good_qs)} -> used {len(picked)}, "
          f"estimated rows={n_rows}")

print("\nTotal estimated Part A rows:",
      sum(p["n_rows_est"] for p in probeA_plan.values()))


## Part A — Extraction: L11 & L1 at the opinion-answer position, batched per question

Only 2 layers are captured (L1, L11) -- far lighter than notebook 13
(which captured all 32 for the sweep). There is no patching at all in
Part A -- pure extraction, so the batch can be 32.

**Results are saved per type** (`probeA_<type>.csv/.npz`) as soon as that
type finishes, and only merged at the end. If the session dies partway,
the types already finished stay safe -- and if this cell is re-run, types
whose files already exist are skipped automatically.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True
)
model.eval()
NUM_LAYERS = model.config.num_hidden_layers
NUM_HEADS = model.config.num_attention_heads
HEAD_DIM = model.config.hidden_size // NUM_HEADS
ALL_HEADS = list(range(NUM_HEADS))

LETTER_IDS = [tokenizer.encode(f" {L}", add_special_tokens=False)[-1] for L in LETTERS]
assert len(set(LETTER_IDS)) == len(LETTER_IDS)

CAPTURE_LAYERS_A = [L1_LAYER, STAR_LAYER]  # Part A: only 2 layers, lightweight

_donor_capture = {}
_capture_positions = []
_active_patch = {}  # reused in Part B

def _oproj_prehook(layer_idx):
    def fn(module, args):
        x = args[0]
        if _capture_positions:
            _donor_capture[layer_idx] = {
                pos: x[:, pos, :].detach().half().cpu() for pos in _capture_positions
            }
        patches = _active_patch.get(layer_idx)
        if patches:
            x = x.clone()
            for (b, pos, heads, alpha, donor) in patches:
                d = donor.to(x.device, x.dtype)
                if len(heads) == NUM_HEADS:
                    x[b, pos, :] = x[b, pos, :] + alpha * (d - x[b, pos, :])
                else:
                    for h in heads:
                        s = slice(h * HEAD_DIM, (h + 1) * HEAD_DIM)
                        x[b, pos, s] = x[b, pos, s] + alpha * (d[s] - x[b, pos, s])
            return (x,) + tuple(args[1:])
        return None
    return fn

# Part A only needs hooks on 2 layers; Part B (the sweep) needs all 32 --
# install ALL of them now (cheap; a hook idles if no capture/patch triggers it)
handles = [model.model.layers[L].self_attn.o_proj.register_forward_pre_hook(_oproj_prehook(L))
           for L in range(NUM_LAYERS)]
print(f"Hooks installed on all {NUM_LAYERS} layers (Part A uses 2, Part B uses all).")

@torch.no_grad()
def forward_batch(prompts, n_opt, capture_positions=None, patch_spec=None):
    global _capture_positions
    _capture_positions = capture_positions or []
    _active_patch.clear()
    if patch_spec:
        _active_patch.update(patch_spec)
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    logits = model(**inputs).logits[:, -1, :]
    _capture_positions = []
    _active_patch.clear()
    selected = logits[:, LETTER_IDS[:n_opt]].float()
    return torch.softmax(selected, dim=1).cpu().numpy()


In [ ]:
BATCH_SIZE = 32   # up from 16: Part A is pure extraction (no patching),
                  # prompts are short & uniform in length, fits on T4x2.

def _save_type(ty, rows, v11, v1):
    """Save the results for ONE type. Called as each type finishes so that a
    session dying partway does not lose everything."""
    pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, f"probeA_{ty}.csv"), index=False)
    np.savez_compressed(os.path.join(OUT_DIR, f"probeA_{ty}.npz"),
                        vecs_l11=np.stack(v11), vecs_l1=np.stack(v1))
    print(f"  [saved] {ty}: {len(rows)} rows -> probeA_{ty}.csv/.npz")

done_types = []
for ty in ALL_TYPES:
    if os.path.exists(os.path.join(OUT_DIR, f"probeA_{ty}.csv")):
        print(f"[skip] {ty} already has results (delete its files to redo it).")
        done_types.append(ty)
        continue

    info = type_opts[ty]
    plan_ty = probeA_plan[ty]
    v1o, v2o = info["v1_opts"], info["v2_opts"]
    rows_ty, vec11_ty, vec1_ty = [], [], []

    for qk in tqdm(plan_ty["good_qs"], desc=f"probeA {ty}"):
        gks_here = [gk for gk in plan_ty["cells"] if qk in info["q_per_cell"].get(gk, set())]
        if len(gks_here) < 5:
            continue
        prompts = [build_prompt(ty, v1o, v2o, gk, qk) for gk in gks_here]
        lens = [len(tokenizer.encode(pr, add_special_tokens=False)) for pr in prompts]
        if len(set(lens)) != 1:
            continue  # rare; skip rather than get the position wrong
        pos_abs = last_token_abs_pos(prompts[0])
        n_opt = len(qmeta[qk][2])

        for start in range(0, len(gks_here), BATCH_SIZE):
            batch_gks = gks_here[start:start + BATCH_SIZE]
            batch_prompts = prompts[start:start + BATCH_SIZE]
            preds = forward_batch(batch_prompts, n_opt, capture_positions=[pos_abs])
            for b, gk in enumerate(batch_gks):
                if (gk, qk) not in real_resp:
                    continue
                rows_ty.append(dict(ty=ty, gk=gk, qk=qk, n_opt=n_opt,
                                    mouth_pred=",".join(f"{x:.6f}" for x in preds[b])))
                # stays fp16 (the hook already does .half()): 4,096 dim x 2 layers x 33k rows
                # = 0.55 GB. The old fp32 version was 4.3 GB -> peak RAM ~8.6 GB during
                # np.stack, prone to OOM right at the last second.
                vec11_ty.append(_donor_capture[STAR_LAYER][pos_abs][b].numpy())
                vec1_ty.append(_donor_capture[L1_LAYER][pos_abs][b].numpy())

    _save_type(ty, rows_ty, vec11_ty, vec1_ty)
    done_types.append(ty)
    del rows_ty, vec11_ty, vec1_ty
    gc.collect()

# --- merge all types into one file (per-type files are kept as a backup) ---
try:
    parts = [pd.read_csv(os.path.join(OUT_DIR, f"probeA_{ty}.csv")) for ty in done_types]
    probeA_df = pd.concat(parts, ignore_index=True)
    probeA_df.to_csv(os.path.join(OUT_DIR, "probe_features_v2.csv"), index=False)
    z = [np.load(os.path.join(OUT_DIR, f"probeA_{ty}.npz")) for ty in done_types]
    np.savez_compressed(os.path.join(OUT_DIR, "probe_features_v2.npz"),
                        vecs_l11=np.concatenate([x["vecs_l11"] for x in z]),
                        vecs_l1=np.concatenate([x["vecs_l1"] for x in z]))
    print(probeA_df.shape, "-> probe_features_v2.csv/.npz")
    print("Unique questions used per type:\n", probeA_df.groupby("ty")["qk"].nunique())
except MemoryError:
    print("Merge failed (memory). That is fine: use the probeA_<type>.csv/.npz files, "
          "and merge them locally.")


## Part B — Extend the 32-layer sweep to 3 new types

The EXACT notebook 13 engine (already passed the offline dry-run): 33
conditions (32 layers + 1 random control) merged into 1 batch per
pair-question.


In [ ]:
N_PAIRS = 12
N_QUESTIONS = 20
MIN_SHARED_Q = 20

rng = np.random.default_rng(RANDOM_SEED)
sweepB_plan = {}
for ty in TYPES_NEW_SWEEP:
    info = type_opts[ty]
    cells = info["cells"]
    q_per_cell = info["q_per_cell"]
    all_pairs = [(a, b) for a in cells for b in cells
                 if a != b and len(q_per_cell[a] & q_per_cell[b]) >= MIN_SHARED_Q]
    pick = rng.choice(len(all_pairs), size=min(N_PAIRS, len(all_pairs)), replace=False)
    pairs = [all_pairs[k] for k in pick]
    pair_questions = {}
    for (a, b) in pairs:
        shared = sorted(q_per_cell[a] & q_per_cell[b])
        wds = sorted(((real_wd(a, b, qk), qk) for qk in shared), reverse=True)
        pair_questions[(a, b)] = [qk for _, qk in wds[:N_QUESTIONS]]
    sweepB_plan[ty] = dict(pairs=pairs, pair_questions=pair_questions)
    print(f"[{ty}] {len(pairs)} pairs")

rng_ctrl = np.random.default_rng(RANDOM_SEED + 7)
_forbidden_ctrl = {(STAR_LAYER, STAR_HEAD)}
while True:
    RAND_HEAD = (int(rng_ctrl.integers(0, NUM_LAYERS)), int(rng_ctrl.integers(0, NUM_HEADS)))
    if RAND_HEAD not in _forbidden_ctrl:
        break
print("Random control head:", RAND_HEAD)


In [ ]:
def wd(pred, real, ordinal):
    return wasserstein_distance(ordinal, ordinal, u_weights=pred, v_weights=real)

CAPTURE_LAYERS_B = list(range(NUM_LAYERS))

# Pass 1: baseline + donors for all layers (batched per question, like the notebook 12 fix)
baselineB_pred = {}
donorsB = {}
id_posB = {}

for ty in TYPES_NEW_SWEEP:
    info = type_opts[ty]
    v1o, v2o = info["v1_opts"], info["v2_opts"]
    needed = sorted({(gk, qk) for (a, b), qs in sweepB_plan[ty]["pair_questions"].items()
                     for qk in qs for gk in (a, b)})
    by_qk = {}
    for (gk, qk) in needed:
        by_qk.setdefault(qk, []).append(gk)

    for qk, gks in tqdm(by_qk.items(), desc=f"baselineB {ty}"):
        prompts = [build_prompt(ty, v1o, v2o, gk, qk) for gk in gks]
        if (ty, qk) not in id_posB:
            id_posB[(ty, qk)] = identity_positions(prompts[0])
        positions = id_posB[(ty, qk)]
        n_opt = len(qmeta[qk][2])
        lens = [len(tokenizer.encode(pr, add_special_tokens=False)) for pr in prompts]
        if len(set(lens)) != 1:
            print(f"  [{ty}/{qk}] lengths not uniform -> skip")
            continue
        for start in range(0, len(gks), 16):
            batch_gks = gks[start:start + 16]
            batch_prompts = prompts[start:start + 16]
            preds = forward_batch(batch_prompts, n_opt, capture_positions=positions)
            for b, gk in enumerate(batch_gks):
                baselineB_pred[(gk, qk)] = preds[b]
                donorsB[(ty, gk, qk)] = {
                    L: {pos: _donor_capture[L][pos][b].clone() for pos in positions}
                    for L in CAPTURE_LAYERS_B
                }
print(f"{len(baselineB_pred)} Part B baselines done.")


In [ ]:
sweepB_rows = []
for ty in TYPES_NEW_SWEEP:
    info = type_opts[ty]
    v1o, v2o = info["v1_opts"], info["v2_opts"]
    for (A, B) in tqdm(sweepB_plan[ty]["pairs"], desc=f"sweepB {ty}"):
        for qk in sweepB_plan[ty]["pair_questions"][(A, B)]:
            if (ty, qk) not in id_posB or (ty, A, qk) not in donorsB or (ty, B, qk) not in donorsB:
                continue
            question, options, ordinal = qmeta[qk]
            n_opt = len(ordinal)
            positions = id_posB[(ty, qk)]
            realA, realB = real_resp[(A, qk)], real_resp[(B, qk)]
            predA, predB = baselineB_pred[(A, qk)], baselineB_pred[(B, qk)]
            prompt_A = build_prompt(ty, v1o, v2o, A, qk)

            n_cond = NUM_LAYERS + 1
            batch_prompts = [prompt_A] * n_cond
            spec = {}
            for L in range(NUM_LAYERS):
                spec.setdefault(L, [])
                for pos in positions:
                    spec[L].append((L, pos, ALL_HEADS, 1.0, donorsB[(ty, B, qk)][L][pos]))
            ctrl_idx = NUM_LAYERS
            rl, rh = RAND_HEAD
            spec.setdefault(rl, [])
            for pos in positions:
                spec[rl].append((ctrl_idx, pos, [rh], 1.0, donorsB[(ty, B, qk)][rl][pos]))

            preds = forward_batch(batch_prompts, n_opt, patch_spec=spec)

            wd_ctrl_realB = wd(preds[ctrl_idx], realB, ordinal)
            wd_A_realA = wd(predA, realA, ordinal)
            wd_A_realB = wd(predA, realB, ordinal)
            wd_B_realB = wd(predB, realB, ordinal)
            for L in range(NUM_LAYERS):
                sweepB_rows.append(dict(
                    attr_type=ty, pair=f"{A} -> {B}", qkey=qk, layer=L,
                    wd_A_to_realA=wd_A_realA, wd_A_to_realB=wd_A_realB,
                    wd_B_to_realB=wd_B_realB,
                    wd_patch_to_realB=wd(preds[L], realB, ordinal),
                    wd_ctrl_to_realB=wd_ctrl_realB,
                ))

sweepB = pd.DataFrame(sweepB_rows)
sweepB["shift_to_realB"] = sweepB["wd_A_to_realB"] - sweepB["wd_patch_to_realB"]
sweepB["shift_ctrl_to_realB"] = sweepB["wd_A_to_realB"] - sweepB["wd_ctrl_to_realB"]
sweepB.to_csv(os.path.join(OUT_DIR, "sweep6_rows.csv"), index=False)
print(sweepB.shape, "-> sweep6_rows.csv (3 new types; max-stat permutation correction is done locally)")


## How to read the results & checklist

**Part A (probe):** after downloading, fit the probe locally (ridge +
LOOCV) from `probe_features_v2.csv/.npz` -- the same as before, but now
each question has far broader cell coverage. Also compare the L1 vs L11
vectors -- is RACE read out better from L1?
The vectors are fp16; cast to float32 before fitting.

**What the paper needs from Part A** (§6.5 "How much is lost in the
corridor" + §8): one number per type -- the probe score from L11/L1 vs the
model's "mouth" score, both against the survey ground truth. The
expectation is a gradation following the fidelity table (finding 06).

**Part B (sweep of 3 new types):** the max-statistic permutation
correction (2000x, same as finding 09) is done LOCALLY from
`sweep6_rows.csv` -- find which layer is significant per type, and set it
beside the finding 06 results (the 6-type fidelity map) to see whether the
"high fidelity -> strong causal use too" pattern also holds for these 3 types.

**Download (MANDATORY):** `probe_features_v2.csv`, `probe_features_v2.npz`,
`sweep6_rows.csv` -> `results/08_probe_causal_dissociation/`.
(If the session is cut short: `probeA_<type>.csv/.npz` are enough too, just
merge them locally.)
